In [1]:
import carla
import random
import numpy as np
import cv2 as cv
import time

In [2]:
client = carla.Client('localhost', 2000)
world = client.get_world()

bpLib = world.get_blueprint_library()
spawnPoints = world.get_map().get_spawn_points()

In [3]:
vehicleBP = bpLib.find('vehicle.lincoln.mkz_2020')
vehicle = world.try_spawn_actor(vehicleBP, random.choice(spawnPoints))
print('vehicle id=', vehicle.id)

spectator = world.get_spectator()
transform = carla.Transform(vehicle.get_transform().transform(carla.Location(x=-4, z=2.5)), carla.Rotation())
spectator.set_transform(transform)

vehicle id= 339


In [4]:
cameraBP = bpLib.find('sensor.camera.rgb')
cameraInitTrans = carla.Transform(carla.Location(x=0.4, z=1.6))
camera = world.spawn_actor(cameraBP, cameraInitTrans, attach_to=vehicle)

In [5]:
def cameraCallBack(image, dataDict):
    dataDict['image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

In [6]:
imageW = cameraBP.get_attribute('image_size_x').as_int()
imageH = cameraBP.get_attribute('image_size_y').as_int()

cameraData = {'image': np.zeros((imageH, imageW, 4))}

camera.listen(lambda image: cameraCallBack(image, cameraData))

In [7]:
vehicle.set_autopilot(True)

In [8]:
cv.namedWindow('RGB Camera', cv.WINDOW_AUTOSIZE)
cv.imshow('RGB Camera', cameraData['image'])
cv.waitKey(1)

while True:
    cv.imshow('RGB Camera', cameraData['image'])

    # print(vehicle.is_alive, vehicle.get_location())

    if cv.waitKey(1) == ord('q'):
        break

cv.destroyAllWindows()